In [1]:
import pandas as pd 

In [2]:
#df_raw = pd.read_csv(("/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260117/"+
#"GRN_gene_embedding_Chiffelle_genes_motif_TFs_raw.csv"), header=0)

# 20260709:
#out_dir = '/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260709/'
#df_raw = pd.read_csv((out_dir+"GRN_gene_embedding_Chiffelle_genes_motif_TFs_raw.csv"), header=0)

# 20260814 batch:
from pathlib import Path

out_dir_root = "/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260814/"

ensemble_mode = 'rankMin_study'
level = 'clonotype'

if level == 'cell':
    sub_d = 'raw_cell_scores'
elif level == 'clonotype':
    sub_d = 'pooled_scores/max'

out_dir = out_dir_root + f'{ensemble_mode}_{level}/' 

df_raw = pd.read_csv((out_dir + "GRN_gene_embedding_Chiffelle_genes_motif_TFs_raw.csv"), header=0)


In [3]:
df_raw

,geneSet,motif,NES,AUC,TF_highConf,TF_lowConf,nEnrGenes,rankAtMax,enrichedGenes
0,geneSet1,taipale__Esrra_DBD_TTCAAGGTCAN_repr,8.84,0.216,ESRRA (inferredBy_Orthology).,ESRRB (inferredBy_MotifSimilarity_n_Orthology).,6,584,CCL3;CCL4;CCL4L2;FGFBP2;GNLY;KLRD1
1,geneSet1,transfac_pro__M01841,5.66,0.144,ESRRA (directAnnotation).,NaN,5,768,CCL3;CCL4;CCL4L2;GZMB;ZNF683
2,geneSet1,transfac_pro__M05700,5.61,0.143,ZNF19 (directAnnotation).,ZFY (inferredBy_MotifSimilarity).,6,1936,CD83;DUSP2;GNLY;GZMH;HLA-DRA;RGS1
3,geneSet1,transfac_pro__M05791,5.41,0.139,ZNF517 (directAnnotation).,NaN,4,431,CCL3;CCL4;CCL4L2;GZMB
4,geneSet1,tfdimers__MD00302,5.24,0.135,IRF7; IRF8 (directAnnotation).,E2F1; IRF1; IRF2; IRF3; IRF4; IRF5; IRF6; MYB;...,11,4615,CCL3;CCL4;CCL4L2;CD83;DUSP2;FGFBP2;GZMA;HLA-DR...
5,geneSet1,tfdimers__MD00602,5.23,0.135,MYB; STAT6 (directAnnotation).,BPTF; E2F6; EP300; IRF1; IRF2; IRF3; IRF4; IRF...,6,1129,CCL3;CCL4;FASLG;IRF4;TNF;TNFRSF18
6,geneSet1,taipale_tf_pairs__TEAD4_CEBPD_NTTRCGYAANNNNNNR...,5.22,0.135,CEBPD; TEAD4 (directAnnotation).,CEBPA; CEBPB; CEBPE; CEBPG; DLX2; DLX3; GCM1; ...,3,189,GADD45B;TNFRSF18;ZNF683
7,geneSet1,transfac_pro__M06897,5.22,0.135,ZNF789 (directAnnotation).,NaN,3,189,CCL4;CCL4L2;GADD45B
8,geneSet1,metacluster_111.4,5.11,0.132,ATF1; ATF1; ATF1; ATF1; ATF1; ATF1; ATF2; ATF2...,ATF1; ATF1; ATF1; ATF1; ATF1; ATF1; ATF1; ATF1...,5,1129,CCL4;CCL4L2;GZMB;IRF4;ZNF683
9,geneSet1,transfac_public__M00285,5.01,0.130,NFE2L1 (directAnnotation).,MYB; TFAP4 (inferredBy_MotifSimilarity).,11,4615,CCL3;CCL4;CCL4L2;FASLG;GADD45B;GZMA;GZMB;IFNG;...


In [4]:
df_raw.TF_highConf.value_counts()

ESRRA (inferredBy_Orthology).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   1
ESRRA (directAnnotation).                                                                                                                                                                                                                                             

In [5]:
import re
import pandas as pd

def parse_tf_highconf(tf_str: str, preferred_method="directAnnotation"):
    """
    Parse strings like:
    "HSF1; HSF1; HSF2 (directAnnotation). HSF1; HSF2; ... (inferredBy_Orthology)."

    Returns a list of TFs, applying rule:
      - If multiple methods exist and preferred_method is present -> keep only that block
      - Else keep TFs from all blocks
    """
    if pd.isna(tf_str) or str(tf_str).strip() == "":
        return []

    s = str(tf_str)

    # Find all blocks: "<genes> (method)"
    # Example match groups:
    #   group1 = "HSF1; HSF1; HSF2"
    #   group2 = "directAnnotation"
    matches = re.findall(r"(.*?)\(([^)]+)\)", s)

    # Fallback: no "(method)" pattern found, treat as simple ';' list
    if not matches:
        genes = [g.strip() for g in s.split(";") if g.strip()]
        return sorted(set(genes))

    blocks = []
    for genes_part, method in matches:
        method = method.strip()

        # Clean gene part: remove trailing dots/spaces
        genes_part = genes_part.strip()
        genes_part = re.sub(r"\.\s*$", "", genes_part)  # trailing "." if present

        # Split TFs by ';'
        tfs = [g.strip() for g in genes_part.split(";") if g.strip()]
        blocks.append((method, tfs))

    methods = {m for m, _ in blocks}

    # Apply your rule
    if len(methods) > 1 and preferred_method in methods:
        tfs = []
        for m, lst in blocks:
            if m == preferred_method:
                tfs.extend(lst)
    else:
        tfs = []
        for _, lst in blocks:
            tfs.extend(lst)

    # Deduplicate while preserving order
    seen = set()
    out = []
    for tf in tfs:
        if tf not in seen:
            out.append(tf)
            seen.add(tf)
    return out


def melt_motif_tf_gene(df: pd.DataFrame,
                       motif_col="motif",
                       tf_col="TF_highConf",
                       genes_col="enrichedGenes",
                       preferred_method="directAnnotation"):
    """
    Return long dataframe with columns: motif, TF, gene
    """
    d = df.copy()

    # Parse TF_highConf into TF list per row
    d["_TF_list"] = d[tf_col].apply(lambda x: parse_tf_highconf(x, preferred_method=preferred_method))

    # Split enrichedGenes into gene list per row
    d["_gene_list"] = d[genes_col].fillna("").apply(
        lambda s: [g.strip() for g in str(s).split(";") if g.strip()]
    )

    # Explode to long format
    long_df = (
        d[[motif_col, "_TF_list", "_gene_list"]]
        .explode("_TF_list")
        .explode("_gene_list")
        .rename(columns={motif_col: "motif", "_TF_list": "TF", "_gene_list": "gene"})
    )

    # Drop empty rows if any
    long_df = long_df.dropna(subset=["TF", "gene"])
    long_df = long_df[(long_df["TF"] != "") & (long_df["gene"] != "")].reset_index(drop=True)

    return long_df


# Example usage:
# out = melt_motif_tf_gene(df)
# out.head()
df_long = melt_motif_tf_gene(df_raw)

In [6]:
df_long

,motif,TF,gene
0,taipale__Esrra_DBD_TTCAAGGTCAN_repr,ESRRA,CCL3
1,taipale__Esrra_DBD_TTCAAGGTCAN_repr,ESRRA,CCL4
2,taipale__Esrra_DBD_TTCAAGGTCAN_repr,ESRRA,CCL4L2
3,taipale__Esrra_DBD_TTCAAGGTCAN_repr,ESRRA,FGFBP2
4,taipale__Esrra_DBD_TTCAAGGTCAN_repr,ESRRA,GNLY
...,...,...,...
316,transfac_pro__M05757,ZNF648,STMN1
317,transfac_pro__M05892,ZNF768,FOS
318,transfac_pro__M05892,ZNF768,HMOX1
319,transfac_pro__M05892,ZNF768,LEF1


In [7]:
{'FOS', 'IRF4', 'LEF1', 'MYC', 'ZNF683'} &  set(df_long.TF.unique())

{'FOS', 'IRF4'}

In [8]:
df_long[df_long.TF == 'FOS']

,motif,TF,gene
228,metacluster_50.1,FOS,ANXA1
229,metacluster_50.1,FOS,CCR7
230,metacluster_50.1,FOS,CD69
231,metacluster_50.1,FOS,DUSP1
232,metacluster_50.1,FOS,HMOX1
233,metacluster_50.1,FOS,KLRG1
234,metacluster_50.1,FOS,TIMP1


In [9]:
# Figure out if the target genes contain the TF as well. 

gene_0_fiPath = out_dir + "GRN_gene_embedding_Chiffelle_genes_celltype_0.csv"

gene_1_fiPath = out_dir + "GRN_gene_embedding_Chiffelle_genes_celltype_1.csv"


gene0 = pd.read_csv(gene_0_fiPath)['genes_celltype_0'].to_list()
gene1 = pd.read_csv(gene_1_fiPath)['genes_celltype_1'].to_list()

In [10]:
# check with the tf database downloaded from https://humantfs.ccbr.utoronto.ca/index.php
out_dir_ = '/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260709/'
tf_database = pd.read_csv(out_dir_ + 'TF_names_v_1.01.txt', header=None)

In [11]:
tf_all = tf_database[0].to_list()

In [12]:
set(gene0+gene1) & set(tf_all)

{'FOS', 'IRF4', 'LEF1', 'MYC', 'ZNF683'}

In [13]:
pd.read_csv(gene_0_fiPath)['genes_celltype_0'].shape

(21,)

In [14]:
len(gene1)

26

In [15]:
# remove exact duplicate motif–TF–gene rows
df_long = df_long.drop_duplicates().reset_index(drop=True)

# save long dataframe
df_long.to_csv((out_dir+
"GRN_gene_embedding_Chiffelle_genes_motif_TFs_long.csv"), index=False)

unique_tfs = (
    df_long["TF"]
    .dropna()
    .astype(str)
    .unique()
)

pd.DataFrame({"TF": unique_tfs}).to_csv(out_dir+
"GRN_gene_embedding_Chiffelle_unique_TFs.csv", index=False)




In [16]:
unique_tfs

array(['ESRRA', 'ZNF19', 'ZNF517', 'IRF7', 'IRF8', 'MYB', 'STAT6',
       'CEBPD', 'TEAD4', 'ZNF789', 'ATF1', 'ATF2', 'ATF3', 'ATF4', 'ATF5',
       'ATF6', 'ATF6B', 'ATF7', 'BATF3', 'CREB1', 'CREB3', 'CREB3L4',
       'CREB5', 'CREM', 'FOSB', 'FOSL1', 'FOSL2', 'JDP2', 'JUN', 'JUNB',
       'JUND', 'XBP1', 'ZNF629', 'NFE2L1', 'ZSCAN2', 'HSF1', 'HSF2',
       'BATF', 'BNC2', 'FOS', 'IRF4', 'SMAD2', 'SMAD3', 'NFE2', 'ZNF283',
       'ZNF549', 'ZNF648', 'ZNF768'], dtype=object)

In [17]:
import os
import time
import pandas as pd
import requests

def download_chip_atlas_targets(
    tf_csv=out_dir+
"GRN_gene_embedding_Chiffelle_unique_TFs.csv",
    out_dir=f"{out_dir}chip_atlas_targets_hg38",
    sleep_sec=0.2,
    timeout=30,
):
    os.makedirs(out_dir, exist_ok=True)

    df = pd.read_csv(tf_csv)
    if "TF" not in df.columns:
        raise ValueError("unique_TFs.csv must have a column named 'TF'.")

    tfs = (
        df["TF"]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s != ""]
        .unique()
        .tolist()
    )

    session = requests.Session()

    not_found = []
    failed = []

    for tf in tfs:
        url = f"https://chip-atlas.dbcls.jp/data/hg38/target/{tf}.10.tsv"
        out_path = os.path.join(out_dir, f"{tf}.10.tsv")

        try:
            r = session.get(url, timeout=timeout)
            if r.status_code == 200:
                with open(out_path, "wb") as f:
                    f.write(r.content)
                print(f"Downloaded: {tf} -> {out_path}")
            elif r.status_code == 404:
                msg = f"Not found on website (404): {tf}"
                print(msg)
                not_found.append(tf)
            else:
                msg = f"Failed: {tf} (HTTP {r.status_code})"
                print(msg)
                failed.append((tf, r.status_code))
        except requests.RequestException as e:
            msg = f"Error downloading {tf}: {e}"
            print(msg)
            failed.append((tf, str(e)))

        time.sleep(sleep_sec)

    # Save summary lists
    if not_found:
        pd.DataFrame({"TF": not_found}).to_csv(os.path.join(out_dir, "not_found_TFs.csv"), index=False)
    if failed:
        pd.DataFrame(failed, columns=["TF", "error"]).to_csv(os.path.join(out_dir, "failed_TFs.csv"), index=False)

    print("\nSummary")
    print(f"Total TFs: {len(tfs)}")
    print(f"Downloaded: {len(tfs) - len(not_found) - len(failed)}")
    print(f"Not found (404): {len(not_found)}")
    print(f"Failed (other): {len(failed)}")

# Run
if __name__ == "__main__":
    download_chip_atlas_targets()


Downloaded: ESRRA -> /cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260814/rankMin_study_clonotype/chip_atlas_targets_hg38/ESRRA.10.tsv
Downloaded: ZNF19 -> /cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260814/rankMin_study_clonotype/chip_atlas_targets_hg38/ZNF19.10.tsv
Not found on website (404): ZNF517
Not found on website (404): IRF7
Downloaded: IRF8 -> /cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260814/rankMin_study_clonotype/chip_atlas_targets_hg38/IRF8.10.tsv
Downloaded: MYB -> /cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260814/rankMin_study_clonotype/chip_atlas_targets_hg38/MYB.10.tsv
Downloaded: STAT6 -> /cluster/pixstor/xudong-lab/suli/Alg_deve

In [18]:
import os
import pandas as pd

def annotate_long_df_with_chip_atlas(
    long_df_path="motif_TF_gene_long.csv",
    targets_dir="chip_atlas_targets_hg38",
    out_path="motif_TF_gene_long_annotated.csv",
):
    # Load long dataframe
    long_df = pd.read_csv(long_df_path)
    required = {"motif", "TF", "gene"}
    missing = required - set(long_df.columns)
    if missing:
        raise ValueError(f"Long dataframe missing columns: {missing}")

    results = []

    # Process per TF to avoid reloading files repeatedly
    for tf, sub in long_df.groupby("TF", sort=False):
        tf_file = os.path.join(targets_dir, f"{tf}.10.tsv")
        if not os.path.exists(tf_file):
            # No file downloaded for this TF, skip
            continue

        try:
            tdf = pd.read_csv(tf_file, sep="\t")
        except Exception as e:
            print(f"Failed to read {tf_file}: {e}")
            continue

        # Check required columns exist
        if "Target_genes" not in tdf.columns:
            print(f"Skip {tf}: missing column 'Target_genes' in {tf_file}")
            continue
        if "STRING" not in tdf.columns:
            print(f"Skip {tf}: missing column 'STRING' in {tf_file}")
            continue

        # second column in the file
        if tdf.shape[1] < 2:
            print(f"Skip {tf}: target file has fewer than 2 columns")
            continue
        second_col_name = tdf.columns[1]

        # Build a compact lookup table for merging
        # Keep only the columns we need, drop duplicates by Target_genes
        lookup = (
            tdf[["Target_genes", second_col_name, "STRING"]]
            .copy()
            .rename(columns={
                "Target_genes": "gene",
                second_col_name: f"TF_average",
                "STRING": "STRING"
            })
        )

        # Normalize gene symbols in both tables to improve matching
        lookup["gene"] = lookup["gene"].astype(str).str.strip()
        sub2 = sub.copy()
        sub2["gene"] = sub2["gene"].astype(str).str.strip()

        # Merge, keep only matches
        merged = sub2.merge(lookup, on="gene", how="inner")



        results.append(merged)

    if results:
        annotated = pd.concat(results, ignore_index=True)
        annotated = annotated.drop_duplicates().reset_index(drop=True)
    else:
        annotated = pd.DataFrame(columns=list(long_df.columns) + ["chipatlas_second_col", "STRING"])

    annotated.to_csv(out_path, index=False)
    print(f"Wrote {len(annotated)} matched rows to: {out_path}")
    return annotated

"""
# Example run
annotated_df = annotate_long_df_with_chip_atlas(
    long_df_path="/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260117/"+
"GRN_gene_embedding_Chiffelle_genes_motif_TFs_long_attn.csv",
    targets_dir="/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260117/chip_atlas_targets_hg38/",
    out_path="/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260117/"+"GRN_gene_embedding_Chiffelle_genes_motif_TFs_long_matched_attn.csv"
)
"""
"""
annotated_df = annotate_long_df_with_chip_atlas(
    long_df_path="/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260117/"+
"GRN_gene_embedding_Chiffelle_genes_motif_TFs_long_attnONLY.csv",
    targets_dir="/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260117/chip_atlas_targets_hg38/",
    out_path="/cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260117/"+"GRN_gene_embedding_Chiffelle_genes_motif_TFs_long_matched_attnONLY.csv"
)
"""

annotated_df = annotate_long_df_with_chip_atlas(
    long_df_path = out_dir+"GRN_gene_embedding_Chiffelle_genes_motif_TFs_long.csv",
    targets_dir=f"{out_dir}chip_atlas_targets_hg38/",
    out_path=f"{out_dir}GRN_gene_embedding_Chiffelle_genes_motif_TFs_long_matched.csv"
)


Wrote 193 matched rows to: /cluster/pixstor/xudong-lab/suli/Alg_development/scRNA_TCRBCR_surfaceProtein/results/classifier/model_explainability/Model_expl_20260814/rankMin_study_clonotype/GRN_gene_embedding_Chiffelle_genes_motif_TFs_long_matched.csv


In [19]:
annotated_df.head()

,motif,TF,gene,TF_average,STRING
0,taipale__Esrra_DBD_TTCAAGGTCAN_repr,ESRRA,CCL3,70.6875,0
1,transfac_pro__M01841,ESRRA,CCL3,70.6875,0
2,taipale__Esrra_DBD_TTCAAGGTCAN_repr,ESRRA,FGFBP2,44.1875,0
3,transfac_pro__M01841,ESRRA,ZNF683,72.6875,0
4,tfdimers__MD00302,IRF8,CCL3,27.3000,0


In [20]:
annotated_df.TF.unique()

array(['ESRRA', 'IRF8', 'MYB', 'CEBPD', 'TEAD4', 'ATF1', 'ATF2', 'ATF3',
       'ATF4', 'ATF6', 'ATF7', 'BATF3', 'CREB1', 'CREB3', 'CREM', 'FOSB',
       'FOSL1', 'FOSL2', 'JUN', 'JUNB', 'JUND', 'XBP1', 'ZNF629',
       'NFE2L1', 'HSF1', 'HSF2', 'BATF', 'BNC2', 'FOS', 'IRF4', 'SMAD2',
       'SMAD3', 'NFE2', 'ZNF768'], dtype=object)

In [21]:
'IRF4' in list(annotated_df.TF.unique())

True

In [22]:
annotated_df[annotated_df.STRING != 0]

,motif,TF,gene,TF_average,STRING
73,metacluster_50.1,FOSL1,TIMP1,81.217391,414
88,metacluster_50.1,JUN,HMOX1,175.546341,401
97,metacluster_50.1,JUNB,CCR7,248.881720,401
112,metacluster_50.1,JUND,HMOX1,197.490566,426
131,metacluster_21.4,HSF1,HSPA1A,500.689076,155
132,metacluster_21.4,HSF1,HSPA1B,601.495798,155
185,metacluster_50.1,SMAD3,TIMP1,91.065217,401


In [23]:
annotated_df.STRING.value_counts()

0      186
401      3
155      2
414      1
426      1
Name: STRING, dtype: int64